# DIVRS — Train PROD trên Colab (ML-10M)

Code: **[HatDuaa/DIVRS-reproduce](https://github.com/HatDuaa/DIVRS-reproduce)**. Luồng: clone → data → train (đủ lâu) → lưu weight+log vào **Drive/Colab Notebooks**.

⚙️ Bật **T4 GPU**: Runtime → Change runtime type → T4 GPU.
⏱️ Train ~30 epoch (~3h). Colab free hay ngắt phiên → cell 6 lưu kết quả ra Drive cho chắc.

## 1. GPU + lib

In [ ]:
!nvidia-smi || echo 'KHONG co GPU -> chay CPU (cham)'
!nproc
!pip install -q absl-py setproctitle deprecated faiss-cpu
import torch; print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())

## 2. Lấy code (hard reset về bản mới nhất GitHub)

In [ ]:
import os
REPO = '/content/DIVRS'
if os.path.exists(REPO):
    !cd {REPO} && git fetch -q origin && git reset --hard -q origin/main
else:
    !git clone -q https://github.com/HatDuaa/DIVRS-reproduce.git {REPO}
%cd {REPO}
!git log --oneline -1

## 3. Tải data ML-10M

In [ ]:
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/ml10m/output/train_coo_record.npz'):
    !cd data && wget -q https://raw.githubusercontent.com/tsinghua-fib-lab/DICE/main/data/ml10m.zip && unzip -oq ml10m.zip
need = ['train_coo_record.npz','val_coo_record.npz','test_coo_record.npz',
        'train_skew_coo_record.npz','popularity.npy','popularity_blend.npy']
miss = [f for f in need if not os.path.exists('data/ml10m/output/'+f)]
print('Data thieu:', miss if miss else 'KHONG — du, san sang!')

## 4. Train PROD + Test
`epochs=50` cap, `es_patience=5` (tự dừng khi val recall bão hòa, thường ~15–30 epoch). Bỏ `--epochs` để dùng mặc định 500.

In [ ]:
import torch
USE_GPU = torch.cuda.is_available()
print('use_gpu =', USE_GPU)
!python app.py \
  --flagfile config/ml10m_DIVRS.cfg \
  --output ./output/ \
  --use_gpu={USE_GPU} --gpu_id=0 \
  --cg_use_gpu=False \
  --num_workers=2 \
  --epochs=50 --es_patience=5

## 5. Xem kết quả (Recall@K / NDCG) của run mới nhất

In [ ]:
import glob
run = max(glob.glob('output/*/'), key=os.path.getmtime)
print('Run moi nhat:', run)
print('===== TEST metrics =====')
!find "{run}test_log" -type f -exec grep -hE "TEST results|recall|hit_ratio|ndcg" {{}} + 2>/dev/null
print('===== VALIDATION theo epoch (recall / iou) =====')
!find "{run}log" -type f -exec grep -hE "VALIDATION" {{}} + 2>/dev/null

## 6. Lưu BEST weight + log vào Drive / Colab Notebooks

In [ ]:
import re, shutil
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/Colab Notebooks/DIVRS_output'
os.makedirs(DRIVE_DIR, exist_ok=True)

run = max(glob.glob('output/*/'), key=os.path.getmtime)
# tìm best epoch theo val recall
best_ep, best_rc = None, -1
for lf in glob.glob(run+'log/*'):
    for m in re.finditer(r"VALIDATION epoch: (\d+).*?'recall': np\.float64\(([\d.]+)\)", open(lf).read()):
        ep, rc = int(m.group(1)), float(m.group(2))
        if rc > best_rc: best_rc, best_ep = rc, ep
print('Best epoch:', best_ep, '| val recall:', round(best_rc,4))

dst = os.path.join(DRIVE_DIR, os.path.basename(run.rstrip('/')))
os.makedirs(dst+'/ckpt', exist_ok=True)
for sub in ['log','test_log']:
    if os.path.isdir(run+sub): shutil.copytree(run+sub, dst+'/'+sub, dirs_exist_ok=True)
ck = run+f'ckpt/epoch_{best_ep}.pth'
if best_ep is not None and os.path.exists(ck):
    shutil.copy(ck, dst+'/ckpt/')
    print('Da copy weight:', ck)
print('XONG -> Drive:', dst)
!ls -lah "{dst}/ckpt/"

## Ghi chú
- **Baseline so sánh** (đúng kiểu paper): đổi `--flagfile` → `config/ml10m_mf.cfg` / `ml10m_ips.cfg` / `ml10m_cause.cfg`.
- **GCN**: `config/ml10m_DIVRS-GCN.cfg` (cần cài `dgl` khớp torch).
- Chậm là do **negative sampler** (CPU-bound) — ~6.5 phút/epoch.
- Paper DIVRS (MF, ML-10M): Recall@20=0.1691, NDCG@20=0.1128 — mục tiêu đối chiếu.